# 4-bit GGUF quantization (Kaggle)

Converts the merged model from `merge_adapter_Kaggle.ipynb` to GGUF and quantizes it to 4 bits
(`Q4_K_M`, ~1GB) with llama.cpp. GGUF is the format Ollama and llama.cpp run on any machine
(NVIDIA, AMD, Apple Silicon or plain CPU).

1. **Build llama.cpp's quantizer**: llama.cpp ships as C++ source; only `llama-quantize` is compiled.
2. **Convert** the merged model to GGUF (bf16) with llama.cpp's Python converter.
3. **Quantize** that GGUF to `Q4_K_M` (the converter alone can't go below 8-bit).
4. **Check quality** on the first 200 test examples, next to the fp32 scores on the same examples
   from `results.md`, so any loss from quantizing is visible.

Output, in the version's Output tab under `quantized-weights/`:
- `commit-model-Q4_K_M.gguf`: the quantized model
- `Modelfile`: for `ollama create`. It sets the plain prompt format the model was trained on;
  without it Ollama would wrap prompts in Qwen's chat template and outputs would degrade.
- `quality_check.json`: the scores and the llama.cpp commit used

**Setup on Kaggle:**
1. Settings -> Accelerator -> GPU T4 (for the quality check only; the rest runs on CPU), Internet -> On
2. Add Data -> attach the output of `merge_adapter_Kaggle.ipynb` (Add Data -> Your Work -> Notebooks)
   and the dataset containing `test.jsonl`
3. Run with **Save Version -> Save & Run All (Commit)**. Building llama.cpp takes a few minutes.

In [ ]:
# --- Dependencies ---
# gguf: lets transformers load the quantized file for the quality check. cmake: builds llama.cpp.
# llama.cpp's own requirements file pins a CPU-only torch, so it's deliberately not installed;
# its converter only needs these extras on top of what Kaggle already has.
%pip install -q -U transformers sentencepiece protobuf gguf sacrebleu cmake "torchao>=0.16.0"

In [ ]:
import glob
import json
import os
import re
import subprocess
import sys

import sacrebleu
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Config ---
OUT_DIR = "/kaggle/working/quantized-weights"
WORK_DIR = "/tmp/quantize"     # intermediates, kept out of the notebook's output
QUANT_TYPE = "Q4_K_M"          # 4-bit; llama.cpp's usual quality/size trade-off
CHECK_SAMPLES = 200            # first N test examples, same ones as the 200-example fp32 run in results.md

SEQ_LEN = 768                  # must match training
MAX_NEW_TOKENS = 40
PROMPT_TEMPLATE = "Write a Conventional Commits message for this diff.\n\n{diff}\n\nCommit message:"

LLAMA_DIR = f"{WORK_DIR}/llama.cpp"
BF16_GGUF = f"{WORK_DIR}/commit-model-bf16.gguf"
QUANT_GGUF = f"{OUT_DIR}/commit-model-{QUANT_TYPE}.gguf"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

def find_input(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Couldn't find {filename} anywhere under /kaggle/input. Check Add Data attached it. "
            f"Found: {glob.glob('/kaggle/input/**/*', recursive=True)[:50]}"
        )
    return matches[0]


def run(cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


def size_gb(path):
    return os.path.getsize(path) / 1e9


# The merged model's folder: the config.json that has safetensors weights next to it.
merged_dirs = [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
    if glob.glob(f"{os.path.dirname(p)}/*.safetensors")
]
if not merged_dirs:
    raise FileNotFoundError("No merged model under /kaggle/input. Attach the output of merge_adapter_Kaggle.ipynb.")
MERGED_DIR = merged_dirs[0]
TEST_FILE = find_input("test.jsonl")
print("Merged model:", MERGED_DIR, sorted(os.listdir(MERGED_DIR)))
print("Test file:   ", TEST_FILE)


# --- 1. Build llama.cpp's quantizer ---
if not os.path.exists(LLAMA_DIR):
    run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA_DIR])
llama_commit = subprocess.run(
    ["git", "-C", LLAMA_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
print("llama.cpp commit:", llama_commit)
run(["cmake", "-S", LLAMA_DIR, "-B", f"{LLAMA_DIR}/build", "-DCMAKE_BUILD_TYPE=Release",
     "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_SERVER=OFF"])
run(["cmake", "--build", f"{LLAMA_DIR}/build", "--target", "llama-quantize", "-j", str(os.cpu_count())])


# --- 2. Convert to GGUF (bf16) ---
run([sys.executable, f"{LLAMA_DIR}/convert_hf_to_gguf.py", MERGED_DIR, "--outfile", BF16_GGUF, "--outtype", "bf16"])


# --- 3. Quantize to 4-bit ---
run([f"{LLAMA_DIR}/build/bin/llama-quantize", BF16_GGUF, QUANT_GGUF, QUANT_TYPE])
print(f"bf16 GGUF: {size_gb(BF16_GGUF):.2f} GB -> {QUANT_TYPE}: {size_gb(QUANT_GGUF):.2f} GB")
os.remove(BF16_GGUF)

# Ollama Modelfile: plain-text prompt (no chat template) and greedy decoding, matching training and
# evaluation. num_ctx leaves headroom above SEQ_LEN: Ollama cuts over-long prompts from the front,
# which would drop the instruction.
with open(f"{OUT_DIR}/Modelfile", "w") as f:
    f.write(f"""FROM ./{os.path.basename(QUANT_GGUF)}
TEMPLATE \"\"\"{{{{ .Prompt }}}}\"\"\"
PARAMETER temperature 0
PARAMETER num_predict {MAX_NEW_TOKENS}
PARAMETER num_ctx 1024
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
""")
print(open(f"{OUT_DIR}/Modelfile").read())


# --- 4. Quality check: does the 4-bit model still score like the fp32 one? ---
# transformers loads the GGUF by dequantizing it, so the 4-bit rounding error is kept. llama.cpp's
# own kernels may differ very slightly, but this catches any real quality loss.
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
tokenizer.padding_side = "left"  # batched generation in a decoder-only model must pad on the left
model = AutoModelForCausalLM.from_pretrained(
    OUT_DIR, gguf_file=os.path.basename(QUANT_GGUF), dtype=torch.float32
).to(device).eval()

TEMPLATE_TOKENS = len(tokenizer(PROMPT_TEMPLATE.format(diff=""), add_special_tokens=False).input_ids)
DIFF_BUDGET = max(SEQ_LEN - TEMPLATE_TOKENS - MAX_NEW_TOKENS - 8, 0)


def fit_prompt(diff):
    # Truncate the diff, not the prompt's tail, so "Commit message:" always survives.
    diff_ids = tokenizer(diff, add_special_tokens=False).input_ids[:DIFF_BUDGET]
    return PROMPT_TEMPLATE.format(diff=tokenizer.decode(diff_ids))


with open(TEST_FILE) as f:
    examples = [json.loads(line) for line in f][:CHECK_SAMPLES]

preds = []
for start in range(0, len(examples), 16):
    batch = examples[start : start + 16]
    inputs = tokenizer([fit_prompt(ex["diff"]) for ex in batch], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    for text in tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1] :], skip_special_tokens=True):
        preds.append((text.strip().splitlines() or [""])[0].strip())

TYPE_RE = re.compile(r"^(\w+)(\(.+?\))?!?:")
VALID_RE = re.compile(r"^(feat|fix|refactor|chore|docs|test|perf|style|build|ci)(\([\w./-]+\))?!?:\s+\S")


def extract_type(message):
    match = TYPE_RE.match(message.strip())
    return match.group(1) if match else None


refs = [ex["message"] for ex in examples]
valid = sum(bool(VALID_RE.match(p)) for p in preds) / len(preds)
type_acc = sum(extract_type(p) == extract_type(r) for p, r in zip(preds, refs)) / len(preds)
bleu = sacrebleu.corpus_bleu(preds, [refs]).score

print(f"\nQuality on the first {len(examples)} test examples:")
print(f"  {'':<16}{'valid format':>14}{'type acc':>10}{'BLEU':>8}")
if CHECK_SAMPLES == 200:
    print(f"  {'fp32 (results)':<16}{0.995:>14.1%}{0.675:>10.1%}{16.67:>8.2f}")
print(f"  {QUANT_TYPE:<16}{valid:>14.1%}{type_acc:>10.1%}{bleu:>8.2f}")

print("\nSample predictions:")
for pred, ref in list(zip(preds, refs))[:10]:
    print(f"  ref:  {ref}\n  pred: {pred}\n")

with open(f"{OUT_DIR}/quality_check.json", "w") as f:
    json.dump({"quant_type": QUANT_TYPE, "llama_cpp_commit": llama_commit, "examples": len(examples),
               "valid_format": valid, "type_acc": type_acc, "BLEU": bleu}, f, indent=2)
print("Output:", sorted(os.listdir(OUT_DIR)))